In [6]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)


LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [7]:
# Output regression data

df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [8]:
# Running R regressions

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/21-policy-imps.R"], check=True, capture_output=True, text=True)
#print(res.stdout)

In [9]:
# Load regression coefficients

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "policy_imps_coefs.parquet"))


In [10]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Embedding Cluster Interaction Effects}
\vspace{0.2cm}
\label{tab_policy_imps}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\multicolumn{5}{c}{Ordered Logit Model} \\
\multicolumn{5}{c}{0=Delayed/Denied, 1=Approved with Conditions, 2=Approved} \\
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table reports coefficient estimates from the ordered logit regression described in Section \ref{sec_methodology}. Dummies for missing values of height and square footage are included for cases without reported physical dimensions. Interaction terms are included to test whether the effect of public opposition, hearing characteristics, and atypicality depend on the embedding cluster. Note that the interaction of consent calendar with embedding cluster 2 is excluded due to multicolinearity with other terms.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1", "r2", "r3","r4"]

vars = [
    ("is_residentialTRUE", "Residential Development"),
    ("is_mixed_useTRUE", "Mixed-Use Development"),
    ("is_nonresidentialTRUE", "Non-Residential Development"),
    ("log_square_footage", "$\\ln$(Square Footage)"),
    ("height", "Height (ft)"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("support_X_cluster1", "$\\ldots \\times$ Embedding Cluster 1"),
    ("support_X_cluster2", "$\\ldots \\times$ Embedding Cluster 2"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)"),
    ("oppose_X_cluster1", "$\\ldots \\times$ Embedding Cluster 1"),
    ("oppose_X_cluster2", "$\\ldots \\times$ Embedding Cluster 2"),
    ("agenda_order", "Agenda Order"),
    ("order_X_cluster1", "$\\ldots \\times$ Embedding Cluster 1"),
    ("order_X_cluster2", "$\\ldots \\times$ Embedding Cluster 2"),
    ("num_agenda_items", "No. Agenda Items"),
    ("is_consent_calendarTRUE", "Consent Calendar"),
    ("consent_calendar_X_cluster1", "$\\ldots \\times$ Embedding Cluster 1"),
    ("atypicality", "Atypicality"),
    ("atypicality_X_cluster1", "$\\ldots \\times$ Embedding Cluster 1"),
    ("atypicality_X_cluster2", "$\\ldots \\times$ Embedding Cluster 2")
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "\n & & & & \\\\ \n"

tbl += "Suffix Group Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="sfx_grp_CUPTRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Council District Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cd_1")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Year Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="yr_2019")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Embedding Cluster Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cluster_fe1TRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"



tbl += "\n & & & & \\\\ \n"

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_policy_imps.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)


\begin{table}[H]
\centering
\caption{Embedding Cluster Interaction Effects}
\vspace{0.2cm}
\label{tab_policy_imps}
\begin{adjustbox}{max height=0.45\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\multicolumn{5}{c}{Ordered Logit Model} \\
\multicolumn{5}{c}{0=Delayed/Denied, 1=Approved with Conditions, 2=Approved} \\
\toprule
 & (1) & (2) & (3) & (4) \\
\midrule
 &  &  &  &  \\
Residential Development  & 0.174$^{}$ & 0.157$^{}$ & 0.068$^{}$ & 0.153$^{}$ \\
 & (0.369) & (0.367) & (0.366) & (0.370) \\ [1.8ex]
Mixed-Use Development  & -0.309$^{}$ & -0.264$^{}$ & -0.369$^{}$ & -0.314$^{}$ \\
 & (0.375) & (0.374) & (0.372) & (0.378) \\ [1.8ex]
Non-Residential Development  & -0.322$^{}$ & -0.299$^{}$ & -0.396$^{}$ & -0.351$^{}$ \\
 & (0.349) & (0.347) & (0.346) & (0.351) \\ [1.8ex]
$\ln$(Square Footage)  & -0.077$^{}$ & -0.066$^{}$ & -0.057$^{}$ & -0.073$^{}$ \\
 & (0.065) & (0.065) & (0.065) & (0.066) \\ [1.8ex]
Height (ft)  & -0.001$^{}$ & -0.000$^{}$ & -0.001$^{}$ & -0.001$^{}$